# AI Slop GAN-style Pipeline (Colab GPU) — **Unsloth + DefAn**

This notebook trains:
- **Discriminator (D):** a lightweight text classifier that scores whether an answer looks *real/definitive* vs *slop*.
- **Generator (G):** a small instruction LLM fine-tuned with **Unsloth** (QLoRA) to generate short definitive answers, then adversarially nudged (SeqGAN-style REINFORCE) to produce **plausible-but-wrong** outputs.

Outputs are saved in **minimal** form:
- Discriminator: standard HF checkpoint folder (can be big).
- Generator: **LoRA adapter only** (small) + tokenizer.

**Dataset:** DefAn definitive-answer QA dataset.


In [ ]:
#@title 0) Install + imports (Colab GPU) — FIXED for Unsloth import order & version compatibility
# Runtime -> Change runtime type -> GPU

!pip -q install -U pip setuptools wheel

# Core libs
!pip -q install -U datasets accelerate evaluate scikit-learn sentencepiece bitsandbytes

# Pin Transformers stack to a known-good range for Unsloth patches (prevents patch NameErrors)
!pip -q install -U "transformers==4.56.2" "trl==0.24.0" "peft==0.12.0"

# Install Unsloth AFTER transformers is pinned (so it patches the right version)
!pip -q install -U unsloth unsloth_zoo

import os, json, random, math, re, gc, pathlib
import numpy as np
import torch

# ✅ IMPORTANT: import unsloth BEFORE transformers/peft usage
import unsloth
from unsloth import FastLanguageModel

from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    set_seed
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("torch:", torch.__version__)
import transformers, trl, peft
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
device: cuda
torch: 2.10.0+cu128
transformers: 4.56.2
trl: 0.24.0
peft: 0.18.1


## 1) Load DefAn dataset

In [ ]:
#@title 1) Download DefAn Public Combined JSON
import urllib.request

DEFAN_URL = "https://raw.githubusercontent.com/ashikiut/DefAn/main/DefAn-Public%20Combined/DefAn_public_combined.json"
data_dir = pathlib.Path("data"); data_dir.mkdir(exist_ok=True)
defan_path = data_dir / "DefAn_public_combined.json"

if not defan_path.exists():
    urllib.request.urlretrieve(DEFAN_URL, defan_path.as_posix())
    print("Downloaded:", defan_path)
else:
    print("Already present:", defan_path)

with open(defan_path, "r", encoding="utf-8") as f:
    raw = json.load(f)

print("records:", len(raw))
print("example:", raw[0])


Downloaded: data/DefAn_public_combined.json
records: 70793
example: {'questions': ' Which team won the 1930 FIFA World Cup?(Give me the name only)', 'answer': 'Uruguay', 'type': 'name', 'domain': 'Sports'}


## 2) Clean + normalize (robust to numeric answers)

In [ ]:
#@title 2) Clean / normalize (FIXED)
def norm_text(x) -> str:
    if x is None:
        return ""
    if isinstance(x, (list, tuple)):
        x = " ".join(str(i) for i in x if i is not None)
    elif isinstance(x, dict):
        x = json.dumps(x, ensure_ascii=False)
    else:
        x = str(x)
    x = x.strip()
    x = re.sub(r"\s+", " ", x)
    return x

records = []
seen = set()
for r in raw:
    q = norm_text(r.get("questions",""))
    a = norm_text(r.get("answer",""))
    t = norm_text(r.get("type","")).lower()
    d = norm_text(r.get("domain",""))
    if not q or not a:
        continue
    key = (q,a,t,d)
    if key in seen:
        continue
    seen.add(key)
    records.append({"question": q, "answer": a, "type": t, "domain": d})

print("clean records:", len(records))
print("example:", records[0])


clean records: 69080
example: {'question': 'Which team won the 1930 FIFA World Cup?(Give me the name only)', 'answer': 'Uruguay', 'type': 'name', 'domain': 'Sports'}


## 3) Build synthetic 'slop' answers

In [ ]:
#@title 3) Slop synthesis helpers
rng = random.Random(7)

pool_by_type = {}
for r in records:
    pool_by_type.setdefault(r["type"], []).append(r["answer"])

def corrupt_numeric(ans: str) -> str:
    nums = re.findall(r"-?\d+", ans.replace(",", ""))
    if not nums:
        return str(rng.randint(1, 9999))
    n = int(nums[0])
    if rng.random() < 0.5:
        delta = max(1, int(abs(n) * rng.uniform(0.01, 0.2)))
        n2 = n + (delta if rng.random() < 0.5 else -delta)
    else:
        n2 = n + rng.randint(-500, 500)
    if n2 == n:
        n2 += 1
    return str(n2)

def corrupt_date(ans: str) -> str:
    s = ans
    m = re.match(r"^(\d{1,2})/(\d{1,2})/(\d{4})$", s)
    if m:
        mm, dd, yy = map(int, m.groups())
        yy2 = yy + rng.choice([-2,-1,1,2,3])
        mm2 = min(12, max(1, mm + rng.choice([-1,0,1])))
        dd2 = min(28, max(1, dd + rng.choice([-3,-1,1,2])))
        return f"{mm2:02d}/{dd2:02d}/{yy2:04d}"
    m = re.match(r"^(\d{4})-(\d{1,2})-(\d{1,2})$", s)
    if m:
        yy, mm, dd = map(int, m.groups())
        yy2 = yy + rng.choice([-2,-1,1,2,3])
        mm2 = min(12, max(1, mm + rng.choice([-1,0,1])))
        dd2 = min(28, max(1, dd + rng.choice([-3,-1,1,2])))
        return f"{yy2:04d}-{mm2:02d}-{dd2:02d}"
    return corrupt_numeric(ans)

def swap_from_pool(ans: str, t: str) -> str:
    pool = pool_by_type.get(t, [])
    if not pool:
        return ans[::-1][: max(1, len(ans)//2)]
    for _ in range(10):
        cand = rng.choice(pool)
        if cand != ans:
            return cand
    return ans

FLUFF = ["", "", " (confirmed).", " — definitely.", " (official).", " according to the records."]

def make_slop_answer(answer: str, t: str) -> str:
    t = (t or "").lower()
    if "num" in t or "rank" in t or "numeric" in t:
        base = corrupt_numeric(answer)
    elif "date" in t:
        base = corrupt_date(answer)
    elif "name" in t or "location" in t or "city" in t or "country" in t:
        base = swap_from_pool(answer, t)
    else:
        base = swap_from_pool(answer, t) if rng.random() < 0.7 else answer
    return (base + rng.choice(FLUFF)).strip()

for r in records[:2]:
    print("Q:", r["question"])
    print("A:", r["answer"])
    print("S:", make_slop_answer(r["answer"], r["type"]))
    print("---")


Q: Which team won the 1930 FIFA World Cup?(Give me the name only)
A: Uruguay
S: Driving Miss Daisy
---
Q: Who emerged as the victor of the 1930 FIFA World Cup?(Give me the name only)
A: Uruguay
S: Clark Gable according to the records.
---


## 4) Discriminator dataset

In [ ]:
#@title 4) Build discriminator dataset
MAX_ROWS = 40000
FAKE_PER_REAL = 1

rng.shuffle(records)
use = records[:MAX_ROWS]

disc_rows = []
for r in use:
    q, a, t = r["question"], r["answer"], r["type"]
    disc_rows.append({"text": f"Q: {q} [SEP] A: {a}", "label": 1})
    for _ in range(FAKE_PER_REAL):
        s = make_slop_answer(a, t)
        disc_rows.append({"text": f"Q: {q} [SEP] A: {s}", "label": 0})

train_rows, val_rows = train_test_split(
    disc_rows, test_size=0.05, random_state=42,
    stratify=[r["label"] for r in disc_rows]
)

disc_ds = DatasetDict({
    "train": Dataset.from_list(train_rows),
    "validation": Dataset.from_list(val_rows),
})
disc_ds


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 76000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 4000
    })
})

## 5) Train Discriminator (robust across transformers versions)

In [ ]:
#@title 5) Discriminator training (robust)
import inspect

disc_model_name = "distilroberta-base"
disc_tok = AutoTokenizer.from_pretrained(disc_model_name)
disc_model = AutoModelForSequenceClassification.from_pretrained(
    disc_model_name, num_labels=2, ignore_mismatched_sizes=True
).to(device)

def disc_tokenize(batch):
    return disc_tok(batch["text"], truncation=True, max_length=256)

remove_cols = [c for c in disc_ds["train"].column_names if c != "label"]
disc_tok_ds = disc_ds.map(disc_tokenize, batched=True, remove_columns=remove_cols)
disc_tok_ds.set_format(type="torch")

collator = DataCollatorWithPadding(tokenizer=disc_tok)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

ta_sig = inspect.signature(TrainingArguments.__init__)
strategy_kw = "eval_strategy" if "eval_strategy" in ta_sig.parameters else "evaluation_strategy"

args_kwargs = dict(
    output_dir="disc_out",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
)
args_kwargs[strategy_kw] = "steps"
args = TrainingArguments(**args_kwargs)

trainer_kwargs = dict(
    model=disc_model,
    args=args,
    train_dataset=disc_tok_ds["train"],
    eval_dataset=disc_tok_ds["validation"],
    data_collator=collator,
    compute_metrics=compute_metrics,
)

tr_sig = inspect.signature(Trainer.__init__)
if "tokenizer" in tr_sig.parameters:
    trainer_kwargs["tokenizer"] = disc_tok
elif "processing_class" in tr_sig.parameters:
    trainer_kwargs["processing_class"] = disc_tok

trainer = Trainer(**trainer_kwargs)
trainer.train()

# Disable notebook progress callback if present (prevents some evaluate crashes)
try:
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)
except Exception:
    pass

print(trainer.evaluate())


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/76000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Accuracy
500,0.309600,0.302056,0.875250
1000,0.311400,0.277329,0.890750
1500,0.263600,0.277091,0.898500
2000,0.257100,0.255499,0.901250
2500,0.246900,0.255696,0.902000
3000,0.255900,0.248455,0.903750
3500,0.254600,0.239220,0.904750
4000,0.248300,0.246991,0.905500
4500,0.261600,0.239471,0.905750


{'eval_loss': 0.2423475682735443, 'eval_accuracy': 0.90575, 'eval_runtime': 1.5967, 'eval_samples_per_second': 2505.193, 'eval_steps_per_second': 78.287, 'epoch': 1.0}


## 6) Generator with Unsloth (QLoRA)

In [ ]:
#@title 6a) Prepare generator dataset (text format)
GEN_MAX_ROWS = 20000
gen_use = use[:GEN_MAX_ROWS]

def format_text(r):
    return f"Q: {r['question']}\nA: {r['answer']}"

gen_texts = [format_text(r) for r in gen_use]
gen_ds = Dataset.from_dict({"text": gen_texts}).train_test_split(test_size=0.05, seed=42)
gen_ds


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 19000
    })
    test: Dataset({
        features: ['text'],
        num_rows: 1000
    })
})

In [ ]:
#@title 6b) Supervised fine-tune Generator with Unsloth (QLoRA)
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

set_seed(42)

max_seq_length = 256
model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"

model, gen_tok = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    max_seq_length = max_seq_length,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = gen_tok,
    train_dataset = gen_ds["train"],
    eval_dataset  = gen_ds["test"],
    dataset_text_field = "text",
    args = SFTConfig(
        output_dir = "gen_unsloth_out",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        max_steps = 600,
        logging_steps = 10,
        save_steps = 200,
        eval_strategy = "no",   # avoids eval/evaluation_strategy drift
        optim = "adamw_8bit",
        lr_scheduler_type = "cosine",
        learning_rate = 2e-4,
        fp16 = False,
        bf16 = True,
        seed = 42,
        report_to = "none",
    ),
)

trainer.train()
FastLanguageModel.for_inference(model)
print("Done SFT.")


==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/19000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 19,000 | Num Epochs = 1 | Total steps = 600
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
10,3.286800
20,2.354600
30,1.819600
40,1.440300
50,1.297500
60,1.221400
70,1.223500
80,1.203400
90,1.120100
100,1.171500


Done SFT.


## 6c) GAN-style adversarial fine-tuning (OOM-safe)

In [ ]:
#@title 6c) REINFORCE adversarial loop (OOM-safe)
from tqdm.auto import tqdm

ADV_STEPS = 200
MAX_NEW_TOKENS = 16
alpha_incorrect = 0.7
beta_len = 0.02

# Freeze discriminator
disc_model.eval()
for p in disc_model.parameters():
    p.requires_grad_(False)

# Train LoRA only
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=2e-5)

def normalize_ans(s):
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    s = s.rstrip(".").strip()
    return s.lower()

ref_map = {r["question"]: r["answer"] for r in use}

@torch.no_grad()
def disc_p_real(question, answer):
    text = f"Q: {question} [SEP] A: {answer}"
    enc = disc_tok(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    logits = disc_model(**enc).logits
    probs = torch.softmax(logits, dim=-1)
    return probs[0,1].item()

def sample_answer_with_logprobs(question):
    prompt = f"Q: {question}\nA:"
    enc = gen_tok(prompt, return_tensors="pt").to(device)
    generated = enc["input_ids"]

    logprob_sum = 0.0
    for _ in range(MAX_NEW_TOKENS):
        out = model(input_ids=generated, use_cache=False)
        logits = out.logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)

        dist = torch.distributions.Categorical(probs=probs)
        next_id = dist.sample()
        logprob_sum = logprob_sum + dist.log_prob(next_id)

        generated = torch.cat([generated, next_id.unsqueeze(0)], dim=1)
        if next_id.item() == getattr(gen_tok, "eos_token_id", None):
            break

    text = gen_tok.decode(generated[0], skip_special_tokens=True)
    ans = text.split("A:", 1)[-1].strip().split("\n")[0].strip()
    return ans, logprob_sum

model.train()
pbar = tqdm(range(ADV_STEPS))
for step in pbar:
    q = use[step % len(use)]["question"]
    ans, logp = sample_answer_with_logprobs(q)
    ref = ref_map[q]

    p_real = disc_p_real(q, ans)
    incorrect = 1.0 if normalize_ans(ans) != normalize_ans(ref) else 0.0
    length_pen = beta_len * max(1, len(ans.split()))
    reward = p_real + alpha_incorrect*incorrect - length_pen

    loss = -(torch.tensor(reward, device=device, dtype=torch.float32) * logp)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
    optimizer.step()

    if step % 10 == 0:
        pbar.set_postfix({"reward": float(reward), "p_real": float(p_real), "incorrect": float(incorrect)})

FastLanguageModel.for_inference(model)
print("Done adversarial loop.")


  0%|          | 0/200 [00:00<?, ?it/s]

Done adversarial loop.


## 7) Demo

In [ ]:
#@title 7) Demo: generate slop + score
import random

def generate_answer(question, temperature=1.0, top_p=0.9, max_new_tokens=16):
    prompt = f"Q: {question}\nA:"
    enc = gen_tok(prompt, return_tensors="pt").to(device)
    out = model.generate(
        **enc,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_new_tokens,
        pad_token_id=getattr(gen_tok, "pad_token_id", None),
        eos_token_id=getattr(gen_tok, "eos_token_id", None),
        use_cache=True,
    )
    text = gen_tok.decode(out[0], skip_special_tokens=True)
    return text.split("A:", 1)[-1].strip().split("\n")[0].strip()

for r in random.sample(use, 8):
    q, ref = r["question"], r["answer"]
    gen_ans = generate_answer(q)
    p_real = disc_p_real(q, gen_ans)
    print("Q:", q)
    print("REF:", ref)
    print("GEN:", gen_ans)
    print("D P(real):", round(p_real, 4), "| incorrect:", normalize_ans(gen_ans) != normalize_ans(ref))
    print("-"*80)


Q: there are 16 teams in a tournament . if during the first round , each team plays every other team exactly once , how many games will be played in the first round ?(Give me the numeric answer only)
REF: 120
GEN: 120(Give me the numeric answer only)
D P(real): 0.9526 | incorrect: True
--------------------------------------------------------------------------------
Q: What was the count of Western Australia's 5-9 age range population in 2016?
REF: 164,153
GEN: 179,661 A (excl. Australian Capital Territory and Northern Territory)
D P(real): 0.9997 | incorrect: True
--------------------------------------------------------------------------------
Q: a courier charges for packages to a certain destination are 65 cents for the first 250 grams and 10 cents for each additional 100 grams or part thereof . what could be the weight in grams of a package for which the charge is $ 1.75 ?(Give me the numeric answer only)
REF: 1280
GEN: 4000.0(Give me the numeric answer only)
D P(real): 0.9996 | inc

## 8) Save minimal generator weights (LoRA adapter)

In [ ]:
#@title 8) Save generator LoRA adapter (small)
import os
save_dir = "gen_lora_adapter"
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
gen_tok.save_pretrained(save_dir)

print("Saved generator LoRA adapter to:", save_dir)
print("Zip for download:")
print("  !zip -r gen_lora_adapter.zip gen_lora_adapter")


Saved generator LoRA adapter to: gen_lora_adapter
Zip for download:
  !zip -r gen_lora_adapter.zip gen_lora_adapter


## 9) Reload later (minimal)

In [ ]:
#@title 9) Reload generator adapter later (example)
# from unsloth import FastLanguageModel
# from peft import PeftModel
# base, tok = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
#     max_seq_length = 256,
#     load_in_4bit = True,
# )
# model = PeftModel.from_pretrained(base, "gen_lora_adapter")
# FastLanguageModel.for_inference(model)
print("See commented code for reload.")
